# EXHEART — lean figure regeneration (MBEC)
Loads the **already-saved results (CSVs) and models** from the repo and redraws figures at 600 dpi + EPS,
in Arial, titles removed. **No training. No dataset download.** Run top to bottom.

Covers 17 figures directly. Fig 10 is provided separately; Fig 1 is your schematic; and Fig 3, 5, 11, 13, 15
are ROC/calibration/interaction/history plots whose raw arrays are not in the CSVs (see the note at the end).

In [1]:
# --- setup: clone repo (data+models+CSVs are committed), install, define save_fig ---
import os
if not os.path.isdir('exheart-research'):
    !git clone -q https://github.com/anasbiswas1/exheart-research.git
%cd exheart-research
!pip install -q shap xgboost lightgbm seaborn
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, seaborn as sns
plt.rcParams['font.family']='sans-serif'
plt.rcParams['font.sans-serif']=['Arial','Liberation Sans','Helvetica','DejaVu Sans']
plt.rcParams['pdf.fonttype']=42; plt.rcParams['ps.fonttype']=42
R='results/'; OUT='submission_figures'; os.makedirs(OUT, exist_ok=True)
def save_fig(fig,n):
    for ax in fig.get_axes(): ax.set_title('')
    try:
        if fig._suptitle is not None: fig._suptitle.set_text('')
    except Exception: pass
    fig.savefig(f'{OUT}/Fig{n}.eps',format='eps',bbox_inches='tight')
    fig.savefig(f'{OUT}/Fig{n}.tiff',dpi=600,bbox_inches='tight',pil_kwargs={'compression':'tiff_lzw'})
    print(f'  saved Fig{n}'); plt.close(fig)

/content/exheart-research


In [2]:
# --- Fig 2: ROC + PR (from saved predictions) ---
from sklearn.metrics import roc_curve,precision_recall_curve,roc_auc_score,average_precision_score
pr=pd.read_csv(R+'brfss2015/brfss2015_test_predictions.csv'); y=pr.y_true.values; p=pr.p_cal.values
auc=roc_auc_score(y,p); ap=average_precision_score(y,p)
fig,ax=plt.subplots(1,2,figsize=(12,5))
fpr,tpr,_=roc_curve(y,p); ax[0].plot(fpr,tpr,color='steelblue',lw=2,label=f'Stack+Platt (AUC={auc:.3f})'); ax[0].plot([0,1],[0,1],'k--',lw=1); ax[0].set_xlabel('False Positive Rate'); ax[0].set_ylabel('True Positive Rate'); ax[0].legend()
pc,rc,_=precision_recall_curve(y,p); ax[1].plot(rc,pc,color='darkorange',lw=2,label=f'AUPRC={ap:.3f}'); ax[1].axhline(y.mean(),color='k',ls='--',lw=1,label='Baseline'); ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision'); ax[1].legend()
plt.tight_layout(); save_fig(fig,'2')

  saved Fig2


In [3]:
# --- Fig 4: Decision Curve Analysis ---
d=pd.read_csv(R+'brfss2015/tables/dca.csv'); fig,ax=plt.subplots(figsize=(7,5))
ax.plot(d.threshold,d.net_benefit_model,lw=2,label='EXHEART'); ax.plot(d.threshold,d.net_benefit_all,'--',label='Treat all'); ax.plot(d.threshold,d.net_benefit_none,'k-',lw=1,label='Treat none')
ax.set_xlabel('Threshold probability'); ax.set_ylabel('Net benefit'); ax.set_ylim(-0.05,0.15); ax.legend(); save_fig(fig,'4')

  saved Fig4


In [4]:
# --- Fig 6 & 14: global SHAP importance bar (from saved ranks) ---
for fign,ds in [('6','brfss2015'),('14','cardio')]:
    s=pd.read_csv(R+f'{ds}/tables/shap_global_ranks.csv').sort_values('mean_abs_shap').tail(15)
    fig,ax=plt.subplots(figsize=(8,6)); ax.barh(s.feature,s.mean_abs_shap,color='steelblue'); ax.set_xlabel('mean(|SHAP value|)'); save_fig(fig,fign)

  saved Fig6
  saved Fig14


In [5]:
# --- Fig 8 & 16: SHAP vs LIME rank scatter (from saved ranks) ---
for fign,ds in [('8','brfss2015'),('16','cardio')]:
    sh=pd.read_csv(R+f'{ds}/tables/shap_global_ranks.csv'); li=pd.read_csv(R+f'{ds}/tables/lime_global_ranks.csv')
    mm=sh.merge(li,on='feature'); fig,ax=plt.subplots(figsize=(6,6)); ax.scatter(mm.shap_rank,mm.lime_rank,color='steelblue')
    ax.plot([1,len(mm)],[1,len(mm)],'k--',alpha=0.3); ax.set_xlabel('SHAP rank'); ax.set_ylabel('LIME rank'); save_fig(fig,fign)

  saved Fig8


  saved Fig16


In [6]:
# --- Fig 9: intersectional TPR heatmap (Sex x Age) ---
it=pd.read_csv(R+'brfss2015/tables/fairness_intersect_sex_age.csv')
piv=it.pivot(index='Sex',columns='Age',values='TPR'); fig,ax=plt.subplots(figsize=(11,2.6))
sns.heatmap(piv,annot=True,fmt='.2f',cmap='RdYlGn',ax=ax,cbar_kws={'label':'TPR'}); ax.set_ylabel('Sex (0=F,1=M)'); save_fig(fig,'9')

  saved Fig9


In [7]:
# --- Fig 12: race/ethnicity TPR + selection rate (BRFSS 2020) ---
r=pd.read_csv(R+'brfss2020/independent_pipeline/tables/fairness_race.csv'); fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].bar(r.race_label,r.TPR,color='steelblue'); ax[0].set_ylabel('TPR'); ax[0].tick_params(axis='x',rotation=45)
ax[1].bar(r.race_label,r.selection_rate,color='seagreen'); ax[1].set_ylabel('Selection rate'); ax[1].tick_params(axis='x',rotation=45); save_fig(fig,'12')

  saved Fig12


In [8]:
# --- Fig 17: Cardio gender + age fairness ---
g=pd.read_csv(R+'cardio/tables/fairness_gender.csv'); a=pd.read_csv(R+'cardio/tables/fairness_age.csv')
fig,ax=plt.subplots(1,2,figsize=(11,4)); ax[0].bar(g.label.astype(str),g.TPR,color='steelblue'); ax[0].set_ylabel('TPR')
ax[1].bar(a.group.astype(str),a.TPR,color='seagreen'); ax[1].set_xlabel('Age group'); ax[1].set_ylabel('TPR'); save_fig(fig,'17')

  saved Fig17


In [9]:
# --- Fig 18: cross-dataset performance summary ---
mp=pd.read_csv(R+'cross_dataset/tables/master_performance.csv'); fig,ax=plt.subplots(figsize=(9,5)); x=range(len(mp))
ax.bar([i-0.2 for i in x],mp.AUC_ROC,0.2,label='AUC-ROC'); ax.bar(list(x),mp.ECE_post,0.2,label='ECE'); ax.bar([i+0.2 for i in x],mp.Sensitivity,0.2,label='Sensitivity')
ax.set_xticks(list(x)); ax.set_xticklabels(mp.Dataset,rotation=25,ha='right'); ax.legend(); save_fig(fig,'18')

  saved Fig18


In [10]:
# --- Fig 19: SHAP rank stability 2015 vs 2020 ---
rs=pd.read_csv(R+'cross_dataset/tables/shap_rank_stability.csv'); fig,ax=plt.subplots(figsize=(7,6))
for _,row in rs.iterrows(): ax.plot([0,1],[row.rank_2015,row.rank_2020],'o-',alpha=0.6)
ax.set_xticks([0,1]); ax.set_xticklabels(['BRFSS 2015','BRFSS 2020']); ax.set_ylabel('SHAP rank'); ax.invert_yaxis(); save_fig(fig,'19')

  saved Fig19


In [11]:
# --- Fig 20: cross-domain SHAP portability slope ---
pt=pd.read_csv(R+'cardio/tables/shap_cross_domain_portability.csv'); fig,ax=plt.subplots(figsize=(7,6))
for _,row in pt.iterrows(): ax.plot([0,1],[row.BRFSS2015_rank,row.Cardio_rank],'o-',alpha=0.6)
ax.set_xticks([0,1]); ax.set_xticklabels(['BRFSS 2015','Cardio']); ax.set_ylabel('SHAP rank'); ax.invert_yaxis(); save_fig(fig,'20')

  saved Fig20


In [12]:
# --- Fig 21: cross-dataset Sex/Gender TPR gap ---
fs=pd.read_csv(R+'cross_dataset/tables/fairness_summary.csv'); row=fs[fs.Metric=='Sex/Gender TPR gap'].iloc[0]
cols=['BRFSS 2015','BRFSS 2020 Transport','BRFSS 2020 Retrained','Cardio']; vals=[float(str(row[cc]).split()[0]) for cc in cols]
fig,ax=plt.subplots(figsize=(8,5)); ax.bar(cols,vals,color=['steelblue','tomato','orange','seagreen']); ax.set_ylabel('Sex/Gender TPR gap'); ax.tick_params(axis='x',rotation=20); save_fig(fig,'21')

  saved Fig21


In [13]:
# --- Fig 22: meta-learner coefficient evolution ---
mc=pd.read_csv(R+'cross_dataset/tables/meta_coef_evolution.csv'); fig,ax=plt.subplots(figsize=(8,5)); x=range(len(mc)); w=0.2
for i,b in enumerate(['XGB','LGBM','RF','MLP']): ax.bar([xx+(i-1.5)*w for xx in x],mc[b],w,label=b)
ax.set_xticks(list(x)); ax.set_xticklabels(mc.Dataset); ax.set_ylabel('Meta-learner coefficient'); ax.legend(); save_fig(fig,'22')

  saved Fig22


In [14]:
# --- Fig 23: SHAP-LIME consistency cross-dataset ---
sc=pd.read_csv(R+'cross_dataset/tables/shap_lime_consistency_comparison.csv'); fig,ax=plt.subplots(figsize=(8,5)); x=range(len(sc))
ax.bar([i-0.15 for i in x],sc.Kendall_tau,0.3,label='Kendall \u03c4'); ax.bar([i+0.15 for i in x],sc.Jaccard_top3,0.3,label='Jaccard@3')
ax.set_xticks(list(x)); ax.set_xticklabels(sc.Dataset); ax.set_ylabel('Value'); ax.legend(); save_fig(fig,'23')

  saved Fig23


In [15]:
# --- Fig S1: operating-threshold sensitivity ---
ts=pd.read_csv(R+'brfss2020/threshold_sensitivity/tables/threshold_sensitivity.csv'); fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(ts.threshold,ts.naive_drop,label='Naive full-model drop'); ax[0].plot(ts.threshold,ts.harm_drift,label='Harmonised drift'); ax[0].set_xlabel('Threshold'); ax[0].set_ylabel('Sensitivity change'); ax[0].legend()
ax[1].plot(ts.threshold,ts.sex_tpr_gap,color='tomato'); ax[1].set_xlabel('Threshold'); ax[1].set_ylabel('Sex TPR gap'); save_fig(fig,'S1')

  saved FigS1


In [16]:
# --- Fig 7: SHAP interaction heatmap (loads saved XGBoost, recomputes - NO training) ---
import shap
from sklearn.model_selection import train_test_split
df=pd.read_csv('data/brfss2015/heart_disease_health_indicators_BRFSS2015.csv')
X=df.drop(columns=['HeartDiseaseorAttack']); yy=df['HeartDiseaseorAttack'].astype(int); FN=X.columns.tolist()
_,Xte,_,_=train_test_split(X,yy,test_size=0.2,random_state=42,stratify=yy)
xgb=joblib.load('models/brfss2015/xgb.pkl')
idx=np.random.RandomState(42).choice(len(Xte),500,replace=False)
inter=np.abs(shap.TreeExplainer(xgb).shap_interaction_values(Xte.values[idx])).mean(0); np.fill_diagonal(inter,0)
fig,ax=plt.subplots(figsize=(11,9)); sns.heatmap(inter,xticklabels=FN,yticklabels=FN,cmap='YlOrRd',ax=ax); save_fig(fig,'7')

  saved Fig7


In [17]:
# --- package for download ---
import shutil
shutil.make_archive('submission_figures','zip','submission_figures')
print('\nDone. Figures in submission_figures/  (also submission_figures.zip)')
print('Generated:', sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download('submission_figures.zip')
except Exception: pass


Done. Figures in submission_figures/  (also submission_figures.zip)
Generated: ['Fig12.eps', 'Fig12.tiff', 'Fig14.eps', 'Fig14.tiff', 'Fig16.eps', 'Fig16.tiff', 'Fig17.eps', 'Fig17.tiff', 'Fig18.eps', 'Fig18.tiff', 'Fig19.eps', 'Fig19.tiff', 'Fig2.eps', 'Fig2.tiff', 'Fig20.eps', 'Fig20.tiff', 'Fig21.eps', 'Fig21.tiff', 'Fig22.eps', 'Fig22.tiff', 'Fig23.eps', 'Fig23.tiff', 'Fig4.eps', 'Fig4.tiff', 'Fig6.eps', 'Fig6.tiff', 'Fig7.eps', 'Fig7.tiff', 'Fig8.eps', 'Fig8.tiff', 'Fig9.eps', 'Fig9.tiff', 'FigS1.eps', 'FigS1.tiff']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Not regenerated here (need model inference the CSVs don't hold)
- **Fig 1** — methodology schematic (drawing tool). Export at 600 dpi yourself.
- **Fig 10** — provided separately (`Fig10.eps` / `Fig10.tiff`).
- **Fig 3** (calibration pre/post curve), **Fig 11** (2020 transport ROC), **Fig 13** (Cardio ROC/PR/calibration),
  **Fig 15** (Cardio SHAP interaction), **Fig 5** (MLP training history): these need the stack predictions,
  the Cardio model, or the per-epoch history, none of which are in the CSVs. Regenerate these five by running
  the corresponding cells in your original notebooks (they **load** the saved models, no retraining), then
  strip the title the same way. Fig 5's history requires re-running only the MLP fit cell.